#MSSA data from HCAI API service

In [0]:
import requests
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder.getOrCreate()

mssa_url = "https://services5.arcgis.com/fMBfBrOnc6OOzh7V/arcgis/rest/services/Medical_Service_Study_Areas_/FeatureServer/0/query?where=1%3D1&outFields=*&outSR=4326&f=json"


mssa_response = requests.get(mssa_url)

print(f"Status code: {mssa_response.status_code}")
# print(f"Content type: {mssa_response.get('content-type')}")

mssa_data = mssa_response.json()


In [0]:
# Use pandas dataframe to flatten the json response
features = mssa_data['features']

mssa_features_df = pd.json_normalize(features)

mssa_features_df.columns

  

Rename the mssa dataframe column labels. 

In [0]:
col_rename_dict = {
    'attributes.FID': 'FID',
    'attributes.STATEFP': 'STATEFP',
    'attributes.COUNTYFP': 'COUNTYFP',
    'attributes.COUNTYNM': 'COUNTYNM',
    'attributes.TRACTCE': 'TRACTCE',
    'attributes.GEOID': 'GEOID',
    'attributes.ALAND': 'ALAND',
    'attributes.AWATER': 'AWATER', 
    'attributes.ASQMI': 'ASQMI',
    'attributes.INTPTLAT': 'INTPTLAT',
    'attributes.INTPTLON': 'INTPTLON',
    'attributes.MSSAID': 'MSSAID',
    'attributes.MSSANM': 'MSSANM',
    'attributes.DEFINITION': 'DEFINITION',
    'attributes.TOTALPOVPO': 'TOTALPOVPO',
    'attributes.Shape__Area': 'Shape_Area', 
    'attributes.Shape__Length': 'Shape_Length',
    'geometry.rings': 'geometry_rings'}


In [0]:
mssa_features_df.rename(columns=col_rename_dict, inplace=True)
mssa_features_df.head()


In [0]:
mssa_features_spark_df = spark.createDataFrame(mssa_features_df)
mssa_features_spark_df.printSchema()

In [0]:
mssa_features_spark_df.write.mode("append").saveAsTable("ca_healthcare_fac_bronze.mssa_data_bronze.mssa_geo")